In [1]:
import re
import joblib
import numpy as np
import pandas as pd

from pathlib import Path
from sklearn.impute import SimpleImputer

from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    brier_score_loss,
    confusion_matrix,
    accuracy_score,
    f1_score
)

In [2]:
PROJECT_ROOT = Path(r"E:\Ph D\Research\iui_ml_prediction")

VALIDATION_FILE = Path(
    r"C:\Users\HP\Desktop\validation.xlsx"
)

REPORT_DIR = PROJECT_ROOT / "reports" / "tables"
MODEL_DIR  = PROJECT_ROOT / "models" / "saved_models"

print(PROJECT_ROOT)
print(VALIDATION_FILE)

E:\Ph D\Research\iui_ml_prediction
C:\Users\HP\Desktop\validation.xlsx


In [3]:
df_val = pd.read_excel(
    VALIDATION_FILE,
    sheet_name="Sheet3"
)

df_val.columns = [
    re.sub(r"[\[\]<>]", "_", str(c)).replace(" ", "_")
    for c in df_val.columns
]

df_val = df_val.loc[
    :,
    ~df_val.columns.str.startswith("Unnamed")
]

df_val = df_val.replace(
    ["NA", "N/A", "na", "n/a", "", " "],
    np.nan
)

print(df_val.shape)
print(df_val.head())

(67, 56)
   Date       HN  Age_Female  Age_Male  Total_infertile_duration  \
0   NaN   402668          32        37                        12   
1   NaN   842930          39        42                         6   
2   NaN  2081297          35        35                        12   
3   NaN  1785356          39        39                        36   
4   NaN  1873868          39        42                         8   

   Pregnancy_History  Number_Of_Alive_Children  Number_Of_Miscarriages  \
0                  0                         0                       0   
1                  1                         0                       1   
2                  1                         0                       1   
3                  0                         0                       0   
4                  1                         0                       1   

   Infertility_Type  Body_Mass_Index  ...  Cycle_Day  Cycle_Number  \
0                 0            29.69  ...         13             1 

In [4]:
df_val["HN"] = df_val["HN"].astype(str).str.strip()

df_val["Result"] = pd.to_numeric(
    df_val["Result"],
    errors="coerce"
)

df_val = df_val.dropna(subset=["Result"])

df_val["Result"] = df_val["Result"].astype(int)

print(df_val["Result"].value_counts())
print("Event rate:", df_val["Result"].mean())

Result
0    63
1     4
Name: count, dtype: int64
Event rate: 0.05970149253731343


In [5]:
pathology_cols = [
    "Uterine_Factors",
    "Tubal_Factors",
    "Ovarian_Factors",
    "Ovulatory_Factors",
    "Cervical_Factors",
    "Endometriosis_Factors",
    "Multisystem_Factors"
]

for c in pathology_cols:
    if c in df_val.columns:
        df_val[c] = pd.to_numeric(
            df_val[c],
            errors="coerce"
        ).fillna(0)

df_val["Total_Female_Pathology"] = (
    df_val[
        [c for c in pathology_cols if c in df_val.columns]
    ].sum(axis=1)
)

df_val["First_TPMSC"] = (
    pd.to_numeric(df_val["First_Volume"], errors="coerce")
    *
    pd.to_numeric(df_val["First_Count"], errors="coerce")
    *
    pd.to_numeric(df_val["First_Progressive_Motile"], errors="coerce")
    / 100
)

df_val["Delta_Motile"] = (
    pd.to_numeric(df_val["Post_Motile"], errors="coerce")
    -
    pd.to_numeric(df_val["Pre_Motile"], errors="coerce")
)

df_val["BMI_InfertilityType_Interaction"] = (
    pd.to_numeric(df_val["Body_Mass_Index"], errors="coerce")
    *
    pd.to_numeric(df_val["Infertility_Type"], errors="coerce")
)

print("Derived features created")

Derived features created


In [6]:
selected_features = pd.read_excel(
    REPORT_DIR /
    "Final_Selected_Features_XGBoost_Baseline.xlsx"
)["Feature"].tolist()

missing_features = [
    c for c in selected_features
    if c not in df_val.columns
]

print("Missing features:")
print(missing_features)

assert len(missing_features) == 0

Missing features:
[]


In [7]:
X_val = df_val[selected_features].copy()

y_val = df_val["Result"].copy()

for c in X_val.columns:
    X_val[c] = pd.to_numeric(
        X_val[c],
        errors="coerce"
    )

print(X_val.shape)

# โหลด development dataset

dev_df = pd.read_csv(
    PROJECT_ROOT /
    "data" /
    "processed" /
    "cycle_level_features.csv"
)

dev_df.columns = [
    re.sub(r"[\[\]<>]", "_", str(c)).replace(" ", "_")
    for c in dev_df.columns
]

X_dev = dev_df[selected_features].copy()

imputer = SimpleImputer(
    strategy="median"
)

imputer.fit(X_dev)

X_val_imputed = pd.DataFrame(
    imputer.transform(X_val),
    columns=selected_features
)

print(
    "Missing after imputation:",
    X_val_imputed.isna().sum().sum()
)

(67, 16)
Missing after imputation: 0


In [8]:
base_model = joblib.load(
    MODEL_DIR /
    "final_model" /
    "XGBoost_Baseline_calibration_base_model.joblib"
)

iso_reg = joblib.load(
    MODEL_DIR /
    "final_model" /
    "isotonic_calibrator_final_xgb.joblib"
)

threshold_df = pd.read_excel(
    REPORT_DIR /
    "calibration_threshold.xlsx"
)

THRESHOLD = float(
    threshold_df["threshold"].iloc[0]
)

prob_raw = (
    base_model
    .predict_proba(X_val_imputed)[:,1]
)

prob_cal = np.clip(
    iso_reg.predict(prob_raw),
    0,
    1
)

y_pred = (
    prob_cal >= THRESHOLD
).astype(int)

roc_auc = roc_auc_score(
    y_val,
    prob_raw
)

pr_auc = average_precision_score(
    y_val,
    prob_raw
)

brier = brier_score_loss(
    y_val,
    prob_cal
)

tn, fp, fn, tp = confusion_matrix(
    y_val,
    y_pred
).ravel()

sens = tp/(tp+fn)
spec = tn/(tn+fp)
npv  = tn/(tn+fn)
ppv  = tp/(tp+fp) if (tp+fp)>0 else 0

print("\n===== TEMPORAL VALIDATION 2026 =====")
print("N =", len(y_val))
print("Positive =", int(y_val.sum()))
print("ROC-AUC =", round(roc_auc,4))
print("PR-AUC =", round(pr_auc,4))
print("Brier =", round(brier,4))
print("Sensitivity =", round(sens,4))
print("Specificity =", round(spec,4))
print("NPV =", round(npv,4))
print("PPV =", round(ppv,4))
print(f"TP={tp} FP={fp} TN={tn} FN={fn}")


===== TEMPORAL VALIDATION 2026 =====
N = 67
Positive = 4
ROC-AUC = 0.4365
PR-AUC = 0.0642
Brier = 0.0584
Sensitivity = 0.0
Specificity = 0.746
NPV = 0.9216
PPV = 0.0
TP=0 FP=16 TN=47 FN=4


In [11]:
df_pred = df_val.copy()

df_pred["Probability_Raw"] = prob_raw
df_pred["Probability_Cal"] = prob_cal
df_pred["Prediction"] = y_pred

In [12]:
df_pred[df_pred["Result"] == 1][
    [
        "HN",
        "Probability_Raw",
        "Probability_Cal",
        "Prediction"
    ]
]

,HN,Probability_Raw,Probability_Cal,Prediction
35,1874733,0.450133,0.045802,0
42,716907,0.291999,0.033898,0
46,2081297,0.519945,0.062500,0
47,2100819,0.363150,0.043478,0


In [18]:
for col in [
    "Age_Female",
    "Post_TPMSC",
    "Cycle_Day",
    "Total_Female_Pathology",
    "First_TPMSC",
]:
    print("\n", col)
    print("Development")
    print(X_dev[col].describe())

    print("Validation")
    print(X_val[col].describe())


 Age_Female
Development
count    2945.000000
mean       35.237691
std         4.160732
min        20.000000
25%        33.000000
50%        35.000000
75%        38.000000
max        68.000000
Name: Age_Female, dtype: float64
Validation
count    67.000000
mean     35.940299
std       3.600844
min      28.000000
25%      33.000000
50%      36.000000
75%      38.000000
max      45.000000
Name: Age_Female, dtype: float64

 Post_TPMSC
Development
count    2945.000000
mean       16.824284
std        17.960063
min         0.000000
25%         3.675339
50%        10.641124
75%        24.250737
max       113.698095
Name: Post_TPMSC, dtype: float64
Validation
count     67.000000
mean      38.514320
std       40.904646
min        0.210000
25%        4.790087
50%       22.704270
75%       58.185114
max      158.367502
Name: Post_TPMSC, dtype: float64

 Cycle_Day
Development
count    2943.000000
mean       14.711859
std         1.855198
min         6.000000
25%        14.000000
50%        14.00000

In [20]:
for col in selected_features:
    print(col, X_val[col].isna().sum())
    

Uterine_Factors 0
Total_Female_Pathology 0
Ovulatory_Factors 0
Cycle_Day 0
First_Count 0
Pre_Count 0
Post_TPMSC 0
Gynecological_Surgical_History 0
Delta_Motile 0
Age_Female 0
First_Volume 1
Post_Count 0
Menstrual_Interval_Days 0
First_Progressive_Motile 0
First_TPMSC 1
BMI_InfertilityType_Interaction 0


In [21]:
df_val[df_val["Result"]==1][
[
"HN",
"Post_TPMSC",
"Cycle_Day",
"Age_Female",
"Uterine_Factors",
"Total_Female_Pathology"
]
]

,HN,Post_TPMSC,Cycle_Day,Age_Female,Uterine_Factors,Total_Female_Pathology
35,1874733,4.012821,12,35,0,0
42,716907,42.314538,14,37,1,1
46,2081297,76.766886,12,36,0,1
47,2100819,12.782880,18,35,1,1


In [22]:
print(base_model.predict_proba(X_dev.iloc[:20])[:,1])

[0.2626928  0.51661205 0.5580404  0.48453    0.25433722 0.25118184
 0.33632812 0.34614727 0.45671546 0.5786931  0.44686696 0.26993474
 0.5255227  0.46201068 0.5422381  0.34740365 0.28661007 0.59277946
 0.33262628 0.43088433]


In [23]:
df_val["Delta_Motile"].describe()
X_dev["Delta_Motile"].describe()
base_model.get_booster().feature_names
base_model.feature_importances_

array([0.17365417, 0.09307659, 0.07692756, 0.03285799, 0.05323429,
       0.04234116, 0.04722229, 0.04650022, 0.02594978, 0.04811624,
       0.04718993, 0.04604716, 0.05995587, 0.05781728, 0.0957225 ,
       0.053387  ], dtype=float32)

In [24]:
print(df_val["Delta_Motile"].describe())
print(X_dev["Delta_Motile"].describe())

count    67.000000
mean     40.951642
std      19.930318
min       7.080000
25%      25.820000
50%      39.200000
75%      50.980000
max      83.480000
Name: Delta_Motile, dtype: float64
count    2945.000000
mean       40.181793
std        20.040666
min       -40.450000
25%        23.800000
50%        37.900000
75%        55.290000
max        97.800000
Name: Delta_Motile, dtype: float64


In [25]:
df_pred[df_pred["Result"] == 1][
    [
        "Age_Female",
        "Post_TPMSC",
        "Delta_Motile",
        "First_TPMSC",
        "BMI_InfertilityType_Interaction",
        "Probability_Raw"
    ]
]

,Age_Female,Post_TPMSC,Delta_Motile,First_TPMSC,BMI_InfertilityType_Interaction,Probability_Raw
35,35,4.012821,39.20,19.540030,0.00,0.450133
42,37,42.314538,37.53,20.013395,0.00,0.291999
46,36,76.766886,7.84,155.151297,25.15,0.519945
47,35,12.782880,46.24,41.331276,30.12,0.363150


In [27]:
import sys
from pathlib import Path

PROJECT_ROOT = Path(r"E:\Ph D\Research\iui_ml_prediction")
sys.path.append(str(PROJECT_ROOT))

In [28]:
import re
import joblib
import numpy as np
import pandas as pd

from pathlib import Path
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    brier_score_loss,
    confusion_matrix,
    accuracy_score,
    f1_score,
)

from src.feature_engineering import (
    add_sperm_wash_features,
    add_cycle_quality_features,
    add_female_interaction_features,
    add_sperm_quality_features,
    add_binary_clinical_flags,
)

# =========================
# PATHS
# =========================

PROJECT_ROOT = Path(r"E:\Ph D\Research\iui_ml_prediction")
VALIDATION_FILE = Path(r"C:\Users\HP\Desktop\validation.xlsx")

REPORT_DIR = PROJECT_ROOT / "reports" / "tables"
MODEL_DIR = PROJECT_ROOT / "models" / "saved_models" / "final_model"
DEV_FILE = PROJECT_ROOT / "data" / "processed" / "cycle_level_features.csv"

# =========================
# LOAD VALIDATION DATA
# =========================

df_val = pd.read_excel(VALIDATION_FILE, sheet_name="Sheet3")

df_val.columns = [
    re.sub(r"[\[\]<>]", "_", str(c)).replace(" ", "_")
    for c in df_val.columns
]

df_val = df_val.loc[:, ~df_val.columns.str.startswith("Unnamed")]

weird_na = ["NA", "N/A", "na", "n/a", "", " ", "-", "nan", "none", "null"]
df_val = df_val.replace(weird_na, np.nan)

print("Raw validation shape:", df_val.shape)

# =========================
# BASIC CLEANING
# =========================

if "HN" in df_val.columns:
    df_val = df_val.dropna(subset=["HN"])
    df_val["HN"] = df_val["HN"].astype(str).str.strip()
    df_val = df_val[df_val["HN"] != ""]

df_val["Result"] = pd.to_numeric(df_val["Result"], errors="coerce")
df_val = df_val.dropna(subset=["Result"])
df_val["Result"] = df_val["Result"].astype(int)

# เหมือน training: ใช้เฉพาะ cycle 1–3
if "Cycle_Number" in df_val.columns:
    df_val["Cycle_Number"] = pd.to_numeric(df_val["Cycle_Number"], errors="coerce")
    df_val = df_val[df_val["Cycle_Number"].isin([1, 2, 3])]

# numeric coercion
for col in df_val.columns:
    if col not in ["HN"]:
        df_val[col] = pd.to_numeric(df_val[col], errors="ignore")

# ถ้า Gynecological_Surgical_History เป็น 0/1 อยู่แล้วก็ไม่กระทบ
if "Gynecological_Surgical_History" in df_val.columns:
    df_val["Gynecological_Surgical_History"] = pd.to_numeric(
        df_val["Gynecological_Surgical_History"],
        errors="coerce"
    )

print("After cleaning:", df_val.shape)
print(df_val["Result"].value_counts())
print("Event rate:", df_val["Result"].mean())

# =========================
# FEATURE ENGINEERING
# ใช้ function เดียวกับ training
# =========================

df_val = add_sperm_wash_features(df_val)
df_val = add_cycle_quality_features(df_val)
df_val = add_female_interaction_features(df_val)
df_val = add_sperm_quality_features(df_val)   # มี clip First_TPMSC upper=200
df_val = add_binary_clinical_flags(df_val)

print("Feature engineering done")

# =========================
# SELECTED FEATURES
# =========================

selected_features = [
    "Uterine_Factors",
    "Total_Female_Pathology",
    "Ovulatory_Factors",
    "Cycle_Day",
    "First_Count",
    "Pre_Count",
    "Post_TPMSC",
    "Gynecological_Surgical_History",
    "Delta_Motile",
    "Age_Female",
    "First_Volume",
    "Post_Count",
    "Menstrual_Interval_Days",
    "First_Progressive_Motile",
    "First_TPMSC",
    "BMI_InfertilityType_Interaction",
]

missing_features = [c for c in selected_features if c not in df_val.columns]
if missing_features:
    raise ValueError(f"Missing selected features in validation data: {missing_features}")

X_val = df_val[selected_features].copy()
y_val = df_val["Result"].copy()

for c in X_val.columns:
    X_val[c] = pd.to_numeric(X_val[c], errors="coerce")

print("X_val shape:", X_val.shape)
print("Missing before imputation:")
print(X_val.isna().sum())

# =========================
# DEVELOPMENT MEDIAN IMPUTER
# =========================

dev_df = pd.read_csv(DEV_FILE)

dev_df.columns = [
    re.sub(r"[\[\]<>]", "_", str(c)).replace(" ", "_")
    for c in dev_df.columns
]

X_dev = dev_df[selected_features].copy()

for c in X_dev.columns:
    X_dev[c] = pd.to_numeric(X_dev[c], errors="coerce")

imputer = SimpleImputer(strategy="median")
imputer.fit(X_dev)

X_val_imputed = pd.DataFrame(
    imputer.transform(X_val),
    columns=selected_features
)

print("Missing after imputation:", X_val_imputed.isna().sum().sum())

# =========================
# LOAD MODEL + CALIBRATOR
# =========================

base_model = joblib.load(
    MODEL_DIR / "XGBoost_Baseline_calibration_base_model.joblib"
)

iso_reg = joblib.load(
    MODEL_DIR / "isotonic_calibrator_final_xgb.joblib"
)

threshold_df = pd.read_excel(
    REPORT_DIR / "calibration_threshold.xlsx"
)

THRESHOLD = float(threshold_df["threshold"].iloc[0])
print("Threshold:", THRESHOLD)

# sanity check feature order
print("Model features:")
print(list(base_model.feature_names_in_))

print("Validation features:")
print(list(X_val_imputed.columns))

if list(base_model.feature_names_in_) != list(X_val_imputed.columns):
    raise ValueError("Feature order mismatch between model and validation data")

# =========================
# PREDICT
# =========================

prob_raw = base_model.predict_proba(X_val_imputed)[:, 1]

prob_cal = np.clip(
    iso_reg.predict(prob_raw),
    0,
    1
)

y_pred = (prob_cal >= THRESHOLD).astype(int)

# =========================
# METRICS
# =========================

roc_auc = roc_auc_score(y_val, prob_raw)
pr_auc = average_precision_score(y_val, prob_raw)
brier = brier_score_loss(y_val, prob_cal)

tn, fp, fn, tp = confusion_matrix(y_val, y_pred).ravel()

sens = tp / (tp + fn) if (tp + fn) > 0 else np.nan
spec = tn / (tn + fp) if (tn + fp) > 0 else np.nan
npv = tn / (tn + fn) if (tn + fn) > 0 else np.nan
ppv = tp / (tp + fp) if (tp + fp) > 0 else np.nan
acc = accuracy_score(y_val, y_pred)
f1 = f1_score(y_val, y_pred, zero_division=0)

print("\n===== TEMPORAL VALIDATION 2026 =====")
print("N =", len(y_val))
print("Positive =", int(y_val.sum()))
print("Event rate =", round(y_val.mean(), 4))
print("ROC-AUC =", round(roc_auc, 4))
print("PR-AUC =", round(pr_auc, 4))
print("Brier =", round(brier, 4))
print("Sensitivity =", round(sens, 4))
print("Specificity =", round(spec, 4))
print("NPV =", round(npv, 4))
print("PPV =", round(ppv, 4))
print("Accuracy =", round(acc, 4))
print("F1 =", round(f1, 4))
print(f"TP={tp} FP={fp} TN={tn} FN={fn}")

# =========================
# SAVE PREDICTIONS
# =========================

df_pred = df_val.copy()
df_pred["Probability_Raw"] = prob_raw
df_pred["Probability_Cal"] = prob_cal
df_pred["Prediction"] = y_pred

out_path = REPORT_DIR / "Temporal_Validation_2026_Predictions.xlsx"
df_pred.to_excel(out_path, index=False)

print("\nSaved predictions to:")
print(out_path)

# ดู positive cases
print("\nPositive cases:")
print(
    df_pred[df_pred["Result"] == 1][
        [
            "HN",
            "Result",
            "Probability_Raw",
            "Probability_Cal",
            "Prediction",
            "Post_TPMSC",
            "First_TPMSC",
            "Delta_Motile",
        ]
    ]
)

Raw validation shape: (67, 56)
After cleaning: (64, 56)
Result
0    60
1     4
Name: count, dtype: int64
Event rate: 0.0625
Feature engineering done
X_val shape: (64, 16)
Missing before imputation:
Uterine_Factors                    0
Total_Female_Pathology             0
Ovulatory_Factors                  0
Cycle_Day                          0
First_Count                        0
Pre_Count                          0
Post_TPMSC                         0
Gynecological_Surgical_History     0
Delta_Motile                       0
Age_Female                         0
First_Volume                       1
Post_Count                         0
Menstrual_Interval_Days            0
First_Progressive_Motile           0
First_TPMSC                        1
BMI_InfertilityType_Interaction    0
dtype: int64
Missing after imputation: 0
Threshold: 0.1009615361690521
Model features:
[np.str_('Uterine_Factors'), np.str_('Total_Female_Pathology'), np.str_('Ovulatory_Factors'), np.str_('Cycle_Day'), np.str_

C:\Users\HP\AppData\Local\Temp\ipykernel_3404\1669780276.py:75: FutureWarning: errors='ignore' is deprecated and will raise in a future version. Use to_numeric without passing `errors` and catch exceptions explicitly instead
  df_val[col] = pd.to_numeric(df_val[col], errors="ignore")


In [29]:
print("\n===== TEMPORAL VALIDATION 2026 =====")
print("N =", len(y_val))
print("Positive =", int(y_val.sum()))
print("Event rate =", round(y_val.mean(), 4))
print("ROC-AUC =", round(roc_auc, 4))
print("PR-AUC =", round(pr_auc, 4))
print("Brier =", round(brier, 4))
print("Sensitivity =", round(sens, 4))
print("Specificity =", round(spec, 4))
print("NPV =", round(npv, 4))
print("PPV =", round(ppv, 4))
print(f"TP={tp} FP={fp} TN={tn} FN={fn}")


===== TEMPORAL VALIDATION 2026 =====
N = 64
Positive = 4
Event rate = 0.0625
ROC-AUC = 0.4333
PR-AUC = 0.0664
Brier = 0.061
Sensitivity = 0.0
Specificity = 0.7333
NPV = 0.9167
PPV = 0.0
TP=0 FP=16 TN=44 FN=4
